## Overview

CNN-GRU Setup for prediction out of 2D timeseries data

finding out of impact of training on "wrong" year of the saison


Colab version of local, loading data from existing dataloaders

----
## Data:

-2D Space - Timeseries

-predicting 1 feature out of 7 variables

-Forecasting 1 timestep (not the following)

-------
Peter Resch, 3.6.

In [ ]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np
import os

from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split

import xarray as xr
rSeed=42

SavePlots = False

## Loading Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


import my_dataloader_module

Mounted at /content/drive/


In [27]:
!cd drive/MyDrive/small_grid && ls

dataloaders			       grid-timeseries_sel_vars_2016.grib.nc
grib_to_pd_dataset.ipynb	       grid-timeseries_sel_vars_2017.grib.nc
grid-timeseries_sel_vars_1940.grib.nc  grid-timeseries_sel_vars_2018.grib.nc
grid-timeseries_sel_vars_1950.grib.nc  grid-timeseries_sel_vars_2019.grib.nc
grid-timeseries_sel_vars_1960.grib.nc  grid-timeseries_sel_vars_2020.grib.nc
grid-timeseries_sel_vars_1970.grib.nc  grid-timeseries_sel_vars_2021.grib.nc
grid-timeseries_sel_vars_1980.grib.nc  grid-timeseries_sel_vars_2022.grib.nc
grid-timeseries_sel_vars_1990.grib.nc  grid-timeseries_sel_vars_2023.grib.nc
grid-timeseries_sel_vars_2000.grib.nc  grid-timeseries_sel_vars_2024.grib.nc
grid-timeseries_sel_vars_2010.grib.nc  grid-timeseries_sel_vars_2025.grib.nc
grid-timeseries_sel_vars_2015.grib.nc


In [ ]:
seasons={"spring": "MAM", "summer": "JJA", "autumn": "SON", "winter": "DJF"}

data_dir='drive/MyDrive/small_grid'
!cd {data_dir} && ls

dataloader_path = str(data_dir+ "/dataloaders/")

dataloaderlist = [p.name for p in Path(dataloader_path).iterdir() if p.is_file()]
#clean dataloaderlist, let only .pt files
dataloaderlist = [f for f in dataloaderlist if f.endswith('.pt')]
dataloaderlist=[f for f in dataloaderlist if "dataloader" in f]      #select only the ones with "dataloader" in the name

['dataloaders\t\t\t       grid-timeseries_sel_vars_2016.grib.nc', 'grib_to_pd_dataset.ipynb\t       grid-timeseries_sel_vars_2017.grib.nc', 'grid-timeseries_sel_vars_1940.grib.nc  grid-timeseries_sel_vars_2018.grib.nc', 'grid-timeseries_sel_vars_1950.grib.nc  grid-timeseries_sel_vars_2019.grib.nc', 'grid-timeseries_sel_vars_1960.grib.nc  grid-timeseries_sel_vars_2020.grib.nc', 'grid-timeseries_sel_vars_1970.grib.nc  grid-timeseries_sel_vars_2021.grib.nc', 'grid-timeseries_sel_vars_1980.grib.nc  grid-timeseries_sel_vars_2022.grib.nc', 'grid-timeseries_sel_vars_1990.grib.nc  grid-timeseries_sel_vars_2023.grib.nc', 'grid-timeseries_sel_vars_2000.grib.nc  grid-timeseries_sel_vars_2024.grib.nc', 'grid-timeseries_sel_vars_2010.grib.nc  grid-timeseries_sel_vars_2025.grib.nc', 'grid-timeseries_sel_vars_2015.grib.nc']
11


## Loading Dataloaders

In [ ]:
dataloaders = {}
i=0
for dl in dataloaderlist:
    time,aim = dl.split("_dataloader_")
    time,season=time.split("_")
    aim = aim.split(".pt")[0]
    #print(time, season, aim)
    dataloaders[time,season,aim] = torch.load(dataloader_path + time + "_" + season + "_dataloader_"+ aim+".pt", weights_only=False)
    i=i+1
print(i,"Dataloaders loaded successfully.")

## Building the Neural Network

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


hidden_size=16

class CNN_GRU(nn.Module):
    def __init__(self,in_channels=7,hidden_size=16,lat_size=5,lon_size=5):
        super().__init__()

        #compressing space
        def double_conv(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True)
            )
        self.conv1 = double_conv(in_channels, 16)
        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))
        self.conv2 = double_conv(16, 32)

        #compressing time
        self.gru = nn.GRU(input_size=32,hidden_size=32*25, num_layers=1,dropout=0.2,batch_first=True)
        
        #expanding space
        self.deconv1 = nn.ConvTranspose2d(32, 16, kernel_size=3, padding=1)
        self.deconv2 = nn.ConvTranspose2d(16, 1, kernel_size=3, padding=1)


    def forward(self, x):
        #print("starting forward:",x.shape)
        x_gru=[]
        #recognize patterns of the spatial data with CNN
        for time in range(x.shape[1]):
            #print(time)
            cnn_in=x[:, time, :, :]
            #print("time,cnn_in.shape:",time,cnn_in.shape)
            cnn_in=self.conv1(cnn_in)
            #print("after conv1:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = self.conv2(cnn_in)
            #print("after conv2:",cnn_in.shape)
            cnn_in = self.maxpool(cnn_in)
            #print("after maxpool:",cnn_in.shape)
            cnn_in = cnn_in.view(cnn_in.size(0), -1)  # Flatten for GRU input
            #print("after flatten:",cnn_in.shape)
            x_gru.append(cnn_in)
        x_gru = torch.stack(x_gru, dim=1)
        #print("shaped for GRU:",x_gru.shape)#(batch, time, convoluted features with space)

        #decoding the temporal patterns with GRU
        x,_ = self.gru(x_gru)#x:(batch, time, hidden_size)
        x = x[:,-1,:].view(x.shape[0],32,5,5)##(batch, last hidden_size,lat_size,lon_size)
        #print("after GRU:",x.shape)
        x=self.deconv1(x)
        #print("after deconv1:",x.shape)
        x=self.deconv2(x)
        #print("after deconv2:",x.shape)
        return x# 


cnn_gru_model=CNN_GRU().to(device)
print(cnn_gru_model)


In [ ]:
def train(dataloader, model, loss_fn, optimizer,device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #print(X.shape, y.shape)
        X, y = X.to(device), y.to(device)
        #print(X.shape, y.shape)
        pred = model(X)#.squeeze()
        #print(pred)#.shape)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"Train Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return loss.item()



def test(dataloader, model, loss_fn,device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            test_loss += loss_fn(predictions, y).item()

            all_predictions.append(predictions.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())

    test_loss /= num_batches
    y_pred = np.concatenate(all_predictions)#[:,:,0]
    y_true = np.concatenate(all_targets)#[:,:,0]
    #print(f"y_true shape: {y_true.shape}, y_pred shape: {y_pred.shape}")

    # Flatten everything to 2D: (samples, features)
    y_true_flat = y_true.reshape(y_true.shape[0], -1)
    y_pred_flat = y_pred.reshape(y_pred.shape[0], -1)

    #mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    #rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    r2 = sklearn.metrics.r2_score(y_true_flat, y_pred_flat)
    print(f"Test Error:\n R2: {r2:>8f}, Avg loss: {test_loss:>8f} \n")
    return test_loss

## Train the model

In [ ]:
time = "recent"

epochs = 100


for season in seasons.keys():
    print(":"*50)
    print(f"Season: {season}")
    model_name = f"cnn_gru_{time}_{season}"

    loss_fcn = nn.MSELoss()
    optimizer_cnn_gru = AdamW(cnn_gru_model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_cnn_gru, mode="min", factor=0.5, patience=2
    )
    train_losses = []
    val_losses = []
    test_losses = []

    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train_loss = train(dataloaders[time, season, "train"], cnn_gru_model, loss_fcn, optimizer_cnn_gru, device)
        val_loss = test(dataloaders[time, season, "val"], cnn_gru_model, loss_fcn, device)
        test_loss = test(dataloaders["now", season, "test"], cnn_gru_model, loss_fcn, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        test_losses.append(test_loss)

        scheduler.step(val_loss)

    train_losses = np.array(train_losses)
    val_losses = np.array(val_losses)
    test_losses = np.array(test_losses)

    print("Training done!")

    os.makedirs("models", exist_ok=True)
    save_path = f"models/{model_name}.pth"

    torch.save({
        "model_state_dict": cnn_gru_model.state_dict(),
        "optimizer_state_dict": optimizer_cnn_gru.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "lag_selection": [0, 24, 48, 60, 66, 69, 71],
        "target_var": 4,
        "epoch": epochs
    }, save_path)

    print(f"Saved model to {save_path}")

    df = pd.DataFrame({
        "Train Loss": train_losses,
        "Validate Loss": val_losses,
        "Test Loss": test_losses
    })
    df.to_csv(f"losses_{model_name}.csv", index=False)
    print(f"Saved losses to losses_{model_name}.csv")

    print(f"Finished traininig {model_name} for {epochs} epochs.")
    print("-"*30)
